# Week 1-2

In [ ]:
import mujoco
import gymnasium as gym
from stable_baselines3 import SAC

env = gym.make("Reacher-v5", xml_file =r"E:\personal projects\FAFO-RL\models\reacher.xml", render_mode = "human")
observation, info = env.reset(seed=42)


print("Obs shape:", observation.shape)
print("Action space:", env.action_space)

try:
    for i in range(1000):  # just 5 steps to see info
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        print(f"Step {i} | reward: {reward:.4f} | info: {info}")

        if terminated or truncated:
            observation, info = env.reset()
finally:
    env.close()

In [ ]:
import mujoco
import gymnasium as gym
from stable_baselines3 import SAC

env = gym.make("Pendulum-v1")
model = SAC("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=20000)
print("Stack working")

In [ ]:
env = gym.make("Pendulum-v1", render_mode = "human")


obs, info = env.reset(seed=42)

try:
    total_reward = 0
    for i in range(1000):  # just 5 steps to see info
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        # print(f"Step {i} | reward: {reward:.4f} | info: {info}")
        total_reward += reward
        if terminated or truncated:
            print(total_reward)
            observation, info = env.reset()
            total_reward = 0
finally:
    env.close()

## Reacher SAC

In [ ]:
import numpy as np
import gymnasium as gym
class CustomPendulumEnv(gym.Wrapper):
    def angle_normalize(self, x):
        return ((x + np.pi) % (2 * np.pi)) - np.pi

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        # obs = [cos(th), sin(th), thdot]
        th = np.arctan2(obs[1], obs[0])   # recover angle from cos/sin
        thdot = obs[2]
        u = np.clip(action, -2.0, 2.0)[0]

        # your custom reward
        costs = self.angle_normalize(th)**2 + 0.1 * thdot**2
        modified_reward = -costs           # dropped action cost term

        return obs, modified_reward, terminated, truncated, info

In [ ]:
import inspect
# import gymnasium as gym

# Create the base environment (unwrapped to bypass time limits/wrappers)
env = CustomPendulumEnv(gym.make("Pendulum-v1"))

# Print the source code of the step function where the reward is calculated
print(inspect.getsource(env.step))

In [ ]:
from stable_baselines3 import SAC, PPO
model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=20e3, tb_log_name="pendulum_modifield_reward")
print("Stack working")

## Different Seed Testing

In [ ]:
# import mujoco
import gymnasium as gym
from stable_baselines3 import SAC, PPO
env = gym.make("Reacher-v5")
# get_device('cpu')
model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=500e3, tb_log_name="reacher_sac_long")
# model = SAC("MlpPolicy", env, verbose=1,seed = 33, device="cpu", tensorboard_log = r".\log")
# model.learn(total_timesteps=20e3, tb_log_name="pendulum_different_seed")
print("Stack working")

In [ ]:
model.save(r".\models\reacher_sac_500k")

In [ ]:
del model

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import SAC

# class MountainCarWrapper(gym.Wrapper):
#     def step(self, action):
#         obs, reward, terminated, truncated, info = self.env.step(action)
#         position, velocity = obs
#         reward += 10 * abs(velocity)
#         reward += 3 * (position + 0.5)
#         return obs, reward, terminated, truncated, info

# env = MountainCarWrapper(gym.make("MountainCarContinuous-v0"))
env = gym.make("MountainCarContinuous-v0")
model = SAC("MlpPolicy", env,
            learning_starts=1000,
            verbose=1,
            tensorboard_log=r".\log",
            seed=42)
model.learn(total_timesteps=100_000,
            tb_log_name="SAC_MountainCar_normal")

In [ ]:
# import mujoco
import gymnasium as gym
from stable_baselines3 import SAC, PPO

model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=100e3, tb_log_name="mountain_car_continuous_reward_wrapper")
print("Stack working")

In [ ]:
# import gymnasium as gym
# from stable_baselines3 import SAC, PPO

env = gym.make("MountainCarContinuous-v0", render_mode = "human")
# model = SAC.load(r".\models\reacher_sac_500k", env=env)

# obs, info = env.reset(seed=32)
obs, info = env.reset()

try:
    total_reward = 0
    for i in range(10000):  # just 5 steps to see info
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        # print(f"Step {i} | reward: {reward:.4f} | info: {info}")
        total_reward += reward
        if terminated or truncated:
        # print(total_reward)
            observation, info = env.reset()
        total_reward = 0
finally:
    env.close()

# Week 3

In [ ]:
import mujoco

model = mujoco.MjModel.from_xml_path(
    r"E:\personal projects\FAFO-RL\agnet_xml\three_pointer.xml"
)
data = mujoco.MjData(model)
print("Bodies  :", model.nbody)
print("Joints  :", model.njnt)
print("Actuators:", model.nu)
print("qpos shape:", data.qpos.shape)
print("Valid ✓")